# Capítulo 7: O que é Aprendizado Estatístico

**Bases 5 — Ciência de Dados** · notebook de aula

Cada célula de código é a mesma do livro e roda na ordem em que aparece — execute de cima para baixo. Versão publicada deste capítulo: [https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/index.html](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/index.html)

> **Gerado automaticamente a partir dos `.qmd` do livro por `scripts/gerar-notebooks.py`.** Edições feitas aqui se perdem no próximo `make notebooks`; para mudar o conteúdo, edite o `.qmd`.

In [ ]:
# Põe o diretório de trabalho na raiz do projeto — é o que faz
# `from scratch...` e os caminhos `dados/...` funcionarem. No livro isso vem
# do `execute-dir: project` do Quarto; aqui é feito à mão.
#
# No Colab não existe cópia do projeto, então esta célula clona uma. É rápido
# (clone raso) e acontece só na primeira execução da sessão.
import os
import subprocess
import sys

REPO = "https://github.com/BragaD/UnDF-Bases5-CienciaDeDados-202602.git"


def raiz_do_projeto(inicio="."):
    """Sobe os diretórios até achar o `_quarto.yml`. None se não houver."""
    atual = os.path.abspath(inicio)
    while not os.path.exists(os.path.join(atual, "_quarto.yml")):
        pai = os.path.dirname(atual)
        if pai == atual:
            return None
        atual = pai
    return atual


raiz = raiz_do_projeto()
if raiz is None:
    destino = "/content/bases5" if os.path.isdir("/content") else "bases5"
    if not os.path.isdir(destino):
        print("baixando o material da disciplina...")
        subprocess.run(["git", "clone", "--depth", "1", REPO, destino], check=True)
    raiz = raiz_do_projeto(destino)

os.chdir(raiz)
if raiz not in sys.path:
    sys.path.insert(0, raiz)

%matplotlib inline
print("diretório de trabalho:", os.getcwd())

> **📌 Nota**
>
> Este capítulo corresponde ao capítulo 2 de James et al. (2023).

> **⚠️ Atenção — Em construção**
>
> A visão geral deste capítulo ainda será escrita.

## Seções

| Seção | Tópico |
|---|---|
| [7.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/01-o-array.html) | O Array |
| [7.2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/02-estimar-f.html) | Estimar f: Predição e Inferência |
| [7.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/03-parametrico-e-nao-parametrico.html) | Paramétrico e Não Paramétrico |
| [7.4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/04-precisao-contra-interpretabilidade.html) | Precisão contra Interpretabilidade |
| [7.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/05-supervisionado-e-nao-supervisionado.html) | Supervisionado e Não Supervisionado |
| [7.6](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/06-qualidade-do-ajuste-e-vies-variancia.html) | Qualidade do Ajuste e o Compromisso Viés-Variância |
| [7.7](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/07-classificacao-e-o-classificador-de-bayes.html) | Classificação e o Classificador de Bayes |

## O Array

> **📌 Nota**
>
> Esta seção corresponde à seção 2.3 de James et al. (2023).

O capítulo anterior fechou com `X` e `y` prontos, os dois ainda como `DataFrame` e `Series` — rótulos de coluna, índice, tudo o que o `pandas` guarda sobre a tabela. A partir daqui o objeto que aparece o tempo todo é outro: todo estimador do `scikit-learn` devolve um `ndarray` — um coeficiente ajustado, uma previsão, uma probabilidade —, e ninguém apresentou esse objeto ainda. Este é o array do `numpy`: o que ele guarda, como se fatia, como se filtra, e o que significa somar "ao longo de um eixo".

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use("estilo-figuras.mplstyle")

### O array, e o tipo único que ele impõe

Um `np.array` nasce de uma lista, mas não herda a flexibilidade dela: uma lista Python guarda qualquer mistura de tipos, um array guarda só um.

In [ ]:
quartos = np.array([2, 3, 1, 4, 2])
quartos, quartos.dtype

`quartos` guarda os mesmos cinco números da lista, mas ganha um atributo que a lista não tem: `dtype`, o tipo único de todo elemento do array — aqui, `int64`. Basta um elemento vir com casa decimal para promover o array inteiro:

In [ ]:
misto = np.array([1, 2, 3.0])
misto.dtype

`misto.dtype` devolve `float64`, não uma mistura de `int64` e `float64` elemento a elemento — os dois primeiros números também viraram ponto flutuante, mesmo tendo entrado como inteiros. Uma lista de listas vira um array de duas dimensões, e `shape` guarda o formato:

In [ ]:
matriz = np.array([[1, 2], [3, 4], [5, 6]])
matriz.shape, matriz.dtype

`(3, 2)` diz três linhas e duas colunas; `dtype`, de novo, é um só para o array inteiro.

### Fatiar sem copiar: a vista

Fatiar um array usa a mesma notação `[início:fim]` de uma lista Python, mas o resultado se comporta diferente.

In [ ]:
original = np.array([10, 20, 30, 40, 50])
fatia = original[1:3]
fatia[0] = 999
original

Alterar `fatia[0]` também mudou `original`. A fatia não é uma cópia dos números: é outra janela sobre o mesmo bloco de memória. É a armadilha número um de quem chega de listas, onde `lista[1:3]` sempre devolve uma lista nova, sem ligação nenhuma com a original. Quando o array de origem precisa continuar intocado, a fatia pede `.copy()`:

In [ ]:
original2 = np.array([10, 20, 30, 40, 50])
copia = original2[1:3].copy()
copia[0] = 999
original2

Desta vez `original2` sai como entrou — `.copy()` força um bloco de memória novo, separado do original.

> **🔷 Conceito**
>
> Fatia (`x[1:3]`) devolve uma **vista**: o mesmo bloco de memória, só enxergado por outra janela. Indexação por **máscara booleana** ou por **lista de posições** (`x[[0, 2]]`) sempre devolve uma **cópia**. `.copy()` força uma cópia em qualquer um dos dois casos, quando o array original precisa ficar fora de alcance.

### A máscara booleana no lugar do laço

Uma comparação entre um array e um número não compara o array inteiro de uma vez: compara elemento a elemento, e devolve um array de `True`/`False` do mesmo tamanho.

In [ ]:
valores = np.array([-3, 5, -1, 8, 0, -7, 2])
mascara = valores > 0
mascara

Usar esse array de booleanos para indexar o array original — `valores[mascara]` — filtra os elementos onde a máscara vale `True`:

In [ ]:
valores[mascara]

É o mesmo resultado que um laço `for` com um `if` dentro produziria, elemento a elemento, só que sem escrever o laço: a máscara descreve a condição uma vez, e o `numpy` aplica sobre o array inteiro.

### A forma de uma redução: `axis`

Somar os elementos de uma matriz aceita um argumento que muda o que "somar" quer dizer: `axis=0` percorre as linhas, coluna por coluna; `axis=1` percorre as colunas, linha por linha. Sobre uma matriz com aluguel e área de quatro imóveis:

In [ ]:
precos = np.array([
    [1200.0, 65.0],
    [800.0, 42.0],
    [2100.0, 98.0],
    [950.0, 55.0],
])
precos.shape

Quatro linhas, duas colunas. Somando ao longo de `axis=0`:

In [ ]:
soma_colunas = precos.sum(axis=0)
soma_colunas.shape, soma_colunas

A forma sai `(2,)` — um total por **coluna**: 5050,0 de aluguel somado, 260,0 de área somada, os quatro imóveis colapsados em uma soma cada. Somando ao longo de `axis=1`:

In [ ]:
soma_linhas = precos.sum(axis=1)
soma_linhas.shape, soma_linhas

A forma agora sai `(4,)` — o oposto: um total por **linha**, aluguel mais área de cada imóvel, os dois números de cada linha colapsados em um só. É a confusão mais comum de quem começa com `axis`: o número que ele nomeia é o eixo que **desaparece** na redução, não o eixo que sobra.

`mean` segue a mesma regra de forma que `sum` — só troca soma por média:

In [ ]:
media_colunas = precos.mean(axis=0)
media_colunas.shape, media_colunas

Forma `(2,)`, de novo uma média por **coluna**: 1262,5 de aluguel médio, 65,0 de área média.

In [ ]:
media_linhas = precos.mean(axis=1)
media_linhas.shape, media_linhas

Forma `(4,)`, uma média por **linha** — um número por imóvel. É a redução que mais volta nos capítulos seguintes: média de coluna para padronizar uma variável, média de linha para resumir uma observação.

### Sorteando com semente: o `rng`

Todo sorteio deste material usa um gerador com semente fixa e explícita, passado adiante em vez de guardado num estado global. Sem semente, cada renderização da página sortearia números diferentes — e as figuras que dependem deles mudariam a cada vez, sem que o texto ao redor mudasse junto.

In [ ]:
rng_a = np.random.default_rng(7)
rng_b = np.random.default_rng(7)
np.array_equal(rng_a.normal(size=3), rng_b.normal(size=3))

Duas instâncias criadas com a mesma semente sorteiam exatamente a mesma sequência — é essa garantia que faz `rng = np.random.default_rng(7)` valer como semente fixa do capítulo inteiro, a partir daqui:

In [ ]:
rng = np.random.default_rng(7)
amostra = rng.normal(size=5)
amostra

`rng.normal` sorteia de uma normal padrão. `rng.choice` sorteia entre valores dados, cada um com a mesma chance por padrão:

In [ ]:
rng.choice(["sim", "não"], size=5)

A figura a seguir usa as duas coisas desta seção ao mesmo tempo: duas coordenadas sorteadas com `rng.normal`, e uma máscara booleana que decide a cor de cada ponto.

In [ ]:
# Figura: Duzentos pontos sorteados com `rng.normal`, coloridos por uma máscara booleana sobre a distância à origem
x = rng.normal(size=200)
y = rng.normal(size=200)
distancia = np.sqrt(x**2 + y**2)
mascara_fig = distancia > 1.5

fig, ax = plt.subplots()
ax.scatter(x[~mascara_fig], y[~mascara_fig], label="até 1,5 da origem")
ax.scatter(x[mascara_fig], y[mascara_fig], label="além de 1,5 da origem")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
mascara_fig.sum(), (~mascara_fig).sum()

55 dos 200 pontos ficam a mais de 1,5 da origem; os outros 145 ficam mais perto — a mesma máscara que filtrou `valores` na seção anterior, agora decidindo a cor de um gráfico em vez de filtrar uma lista de números.

### Do `DataFrame` para o array

O capítulo anterior separou `alugueis` em `X`, os preditores, e `y`, o alvo — os dois como objetos do `pandas`. Aqui `X` volta reduzido às quatro colunas numéricas que já vinham prontas — sem os cinco `dummies` de cidade que a seção 6.6 acrescenta depois —, só para a linha caber numa página:

In [ ]:
alugueis = pd.read_csv("dados/alugueis.csv", na_values=["-"])
X = alugueis[["area_m2", "quartos", "banheiros", "vagas"]]
y = alugueis["aluguel"]
type(X), X.shape, type(y), y.shape

`.to_numpy()` devolve o que está por baixo de cada um: só os números, sem rótulo de coluna nem índice.

In [ ]:
X_array = X.to_numpy()
y_array = y.to_numpy()
type(X_array), X_array.shape, X_array.dtype, type(y_array), y_array.shape, y_array.dtype

Mesmas formas, `(10692, 4)` e `(10692,)`, mas o `DataFrame` virou `ndarray` e a `Series` virou `ndarray` também — e junto com o tipo foram embora os nomes de coluna e o índice. A primeira linha de `X_array`,

In [ ]:
X_array[0]

chega como quatro números soltos — 70, 2, 1, 1 — sem dizer mais qual era área, qual era quarto, qual era banheiro, qual era vaga; só a ordem em que as colunas foram escolhidas continua carregando esse significado. É o mesmo tipo de objeto, sem nome nenhum de coluna, que sai do outro lado de um estimador do `scikit-learn`: um coeficiente ajustado, uma previsão, uma probabilidade. A próxima seção volta à pergunta que abre o capítulo: o que significa estimar uma função a partir de dado, e o que muda entre prever e explicar.

## Estimar f: Predição e Inferência

> **📌 Nota**
>
> Esta seção corresponde às seções 2.1 e 2.1.1 de James et al. (2023).

A seção anterior fechou perguntando o que significa estimar uma função a partir de dado, e o que muda entre prever e explicar. É essa pergunta que abre o ISLP, com o exemplo que também abre esta seção: `Advertising`, o investimento em TV, rádio e jornal de duzentos mercados, contra as vendas de cada um.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use("estilo-figuras.mplstyle")

### A formulação: Y = f(X) + ε

In [ ]:
propaganda = pd.read_csv("dados/Advertising.csv")
propaganda.shape, propaganda.columns.tolist()

Duzentos mercados e quatro colunas: `tv`, `radio` e `jornal`, o investimento em cada mídia, e `vendas`, o volume vendido.

Em notação, `X` = (tv, rádio, jornal) são os preditores — o que se mede e, em alguma medida, se controla —, e `Y` = vendas é a resposta, o que se quer prever ou explicar a partir de `X`. A suposição central deste livro inteiro é que existe uma relação entre os dois, e que ela se escreve como

$$Y = f(X) + \epsilon$$

`f` é uma função fixa, mas desconhecida, de `X`: é a informação sistemática que `X` carrega sobre `Y` — o que aconteceria, em média, se todo mercado com o mesmo investimento em propaganda vendesse a mesma quantidade. ε é o erro: tudo que influencia `Y` e não está em `X` — sazonalidade que a tabela não registra, um concorrente que baixou preço na mesma semana, o próprio ruído de medir "vendas" —, e por definição não pode ser previsto a partir de `X`. ε não é descuido de quem coletou o dado: é a distância entre o que `X` explica e tudo o mais que também importa. Por construção, ε é independente de `X` e tem média zero — não há tendência sistemática de errar para cima nem para baixo, só a parte que `X` não alcança.

### Prever ou explicar: duas perguntas diferentes

Uma vez que existe `f`, o que se faz com ela depende da pergunta. "Quanto vou vender com um orçamento assim?" é uma pergunta de **predição**: o que importa é acertar o valor de `Y`, e a forma exata de `f` pode ficar escondida — um modelo tratado como caixa-preta serve, desde que a previsão saia certa. "O que faz vender mais?" é uma pergunta de **inferência**: aqui a forma de `f` é o que se quer, porque a resposta está nela — quais mídias têm associação com as vendas, se o efeito é positivo ou negativo, se um investimento pequeno em jornal já esgota o que jornal tem a contribuir. Nesse sentido, "inferência" aqui é sobre *entender* a relação entre `X` e `Y`, não sobre calcular um erro-padrão ou um valor-p — essa segunda coisa é assunto de outra disciplina.

Um modelo pode responder bem a uma das duas perguntas e mal à outra, mesmo sobre o mesmo par (`X`, `Y`): uma caixa-preta pode prever vendas com folga e não dizer nada sobre qual mídia cortar num corte de orçamento; um modelo simples o bastante para se ler o efeito de cada mídia pode prever pior do que um mais flexível. Prever e explicar não são a mesma competência.

### Erro redutível e irredutível: o teto de qualquer modelo

Nenhuma estimativa `f̂` acerta `f` perfeitamente, e essa distância chama-se erro **redutível** — redutível porque escolher um método melhor, mais dado ou mais preditores pode encolhê-la. Mas mesmo que `f̂` fosse `f`, ponto a ponto, prever `Y` a partir de `X` ainda erraria: `Y` também depende de ε, que por definição não pode ser previsto a partir de `X`. Esse é o erro **irredutível**, e a distinção fica explícita ao decompor o erro quadrático médio da previsão `Ŷ = f̂(X)`:

$$
E\left[(Y - \hat{Y})^2\right] = \underbrace{\left[f(X) - \hat{f}(X)\right]^2}_{\text{redutível}} + \underbrace{\mathrm{Var}(\epsilon)}_{\text{irredutível}}
$$

O primeiro termo é o que um modelo melhor reduz. O segundo não muda com o modelo — é propriedade do problema, não do método — e por isso funciona como um teto: nenhum ajuste, por melhor que seja, empurra o erro esperado abaixo dele. Na prática o irredutível quase nunca é conhecido de antemão; mais adiante neste capítulo ele volta como o piso que explica por que a curva de erro de teste nunca chega a zero.

### Income1: um f verdadeiro, porque o dado é simulado

Na prática, `f` nunca aparece — só o dado observado, gerado por uma `f` desconhecida mais ε. `Income1` é a exceção que o ISLP usa de propósito: os pontos não vêm de pesquisa real, foram simulados pelos próprios autores, então a função que os gerou é conhecida por construção. É isso que torna possível desenhar `f` ao lado dos pontos, e não só os pontos.

In [ ]:
estudo_renda = pd.read_csv("dados/Income1.csv")
estudo_renda.shape, estudo_renda.columns.tolist()

Trinta indivíduos, anos de estudo (`escolaridade`) e renda em milhares de dólares (`renda`).

A função exata que os autores usaram para simular esses pontos não está disponível fora do pacote `ISLP` do R, que este material não instala. A curva a seguir não é aquela função: é uma média móvel sobre os pontos, ordenados por anos de estudo — uma estimativa suave de `f`, boa o bastante para mostrar a forma da relação, mas uma estimativa, não a verdade que gerou o dado.

In [ ]:
# Figura: Renda contra anos de estudo em `Income1`. A curva é uma estimativa suave de f; cada segmento liga um ponto observado à curva, e esse segmento é o ε daquela observação.
educacao = estudo_renda["escolaridade"].to_numpy()
renda_observada = estudo_renda["renda"].to_numpy()
f_suave = estudo_renda["renda"].rolling(window=13, center=True, min_periods=1).mean().to_numpy()

fig, ax = plt.subplots()
ax.vlines(
    educacao,
    np.minimum(renda_observada, f_suave),
    np.maximum(renda_observada, f_suave),
    color="C1",
    linewidth=1,
)
ax.plot(educacao, f_suave, color="C0", linewidth=2, label="f (estimativa suave)")
ax.scatter(educacao, renda_observada, color="C2", zorder=3, label="observado")
ax.set_xlabel("anos de estudo")
ax.set_ylabel("renda (milhares de dólares)")
ax.legend()
plt.tight_layout()
plt.show()

Os segmentos são o ε de cada observação: a distância entre o que a curva estima para aqueles anos de estudo e o que aquele indivíduo de fato ganha.

In [ ]:
residuo = renda_observada - f_suave
acima = int((residuo > 0).sum())
abaixo = int((residuo < 0).sum())
media = round(float(residuo.mean()), 2)
acima, abaixo, media

Dezesseis pontos ficam acima da curva e catorze abaixo, com a média dos resíduos em 0,10 — perto de zero, não exatamente zero, porque a curva usada aqui é uma estimativa de `f`, não a função que de fato gerou o dado. Nenhum modelo apaga esses segmentos: mesmo com a `f` exata em mãos — o caso raro que a simulação permite —, prever a renda de um indivíduo específico ainda erraria pelo tamanho do seu ε. É esse piso, e não uma falha de ajuste, que a próxima seção começa a explorar ao perguntar como, afinal, se estima `f`.

## Paramétrico e Não Paramétrico

> **📌 Nota**
>
> Esta seção corresponde à seção 2.1.2 de James et al. (2023).

A seção anterior fechou perguntando como, afinal, se estima `f` — a partir só do que se observa, sem acesso à função que gerou o dado. Há duas respostas bem diferentes para essa pergunta, e a diferença entre elas está inteira numa escolha que se faz antes de olhar qualquer ponto: assumir uma forma para `f`, ou deixar o dado decidir a forma sozinho.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.neighbors import KNeighborsRegressor

plt.style.use("estilo-figuras.mplstyle")

### O caminho paramétrico: assumir uma forma

O caminho paramétrico começa assumindo uma forma para `f` — uma reta, um plano, qualquer função que se escreva com um número fixo e pequeno de parâmetros — e troca o problema de estimar uma função inteira, livre para assumir qualquer formato, pelo problema bem mais simples de estimar esses poucos números. A vantagem é direta: poucos parâmetros pedem pouco dado, porque cada ponto observado ajuda a fixar todos eles ao mesmo tempo. O risco mora na mesma decisão: se a forma escolhida não é parecida com a `f` verdadeira, nenhuma quantidade de dado, nem cuidado nenhum no ajuste, conserta — o erro já está embutido na forma, antes mesmo de o primeiro ponto entrar na conta.

`Income2`, outro conjunto simulado do ISLP, dá o exemplo: a renda de trinta pessoas contra dois preditores, não um só como em `Income1` na seção anterior.

In [ ]:
renda2 = pd.read_csv("dados/Income2.csv")
renda2.shape, renda2.columns.tolist()

Trinta linhas, três colunas: `escolaridade` e `senioridade` são os preditores, `renda` é a resposta, em milhares de dólares. Com dois preditores, a forma paramétrica mais simples deixa de ser uma reta e passa a ser um plano — renda como combinação linear de escolaridade e senioridade.

In [ ]:
X = renda2[["escolaridade", "senioridade"]]
y = renda2["renda"]

plano = LinearRegression().fit(X, y)
[round(float(coeficiente), 3) for coeficiente in plano.coef_], round(float(plano.intercept_), 2)

O `LinearRegression` do `scikit-learn` ajusta exatamente essa forma: um plano com coeficiente 5,896 para escolaridade, 0,173 para senioridade, e intercepto -50,09. Três números — só três — descrevem a superfície inteira. Não importa se o dado tivesse trinta pontos ou trinta mil: mais dado deixaria a estimativa mais precisa, mas não mudaria quantos parâmetros o modelo tem.

### O caminho não paramétrico: deixar o dado dizer

O caminho não paramétrico não assume forma nenhuma para `f`: em vez de resumir a relação em poucos números, deixa a vizinhança de cada ponto decidir o valor previsto ali. *k* vizinhos mais próximos (k-NN) é o exemplo mais simples da família: para prever a renda de alguém com uma combinação de escolaridade e senioridade, ele procura as pessoas mais parecidas no dado observado e tira a média das rendas delas — sem supor que a relação seja um plano, uma curva suave ou qualquer coisa com nome fechado. A vantagem é acertar formas que o plano erraria de largada, porque nada na conta impõe retidão a `f`; o custo é precisar de muito mais dado para que essa liberdade compense e, se a flexibilidade for longe demais, ajustar não à relação verdadeira, mas ao ruído específico das trinta pessoas que compõem esta amostra.

In [ ]:
X.agg(["mean", "std"]).round(1)

Escolaridade tem média 16,4 anos e desvio padrão 3,8; senioridade, média 93,9 e desvio padrão 55,7 — uma dispersão bem maior, na mesma unidade de anos. k-NN decide "vizinho mais próximo" por distância, e com as duas colunas nessa escala tão diferente, a distância sem ajuste seria dominada pela que varia mais — a senioridade —, deixando a escolaridade quase sem voz na conta. A padronização resolve isso subtraindo a média e dividindo pelo desvio padrão de cada coluna:

In [ ]:
padronizado = (X - X.mean()) / X.std()
flexivel = KNeighborsRegressor(n_neighbors=3).fit(padronizado, y)

mse_plano = mean_squared_error(y, plano.predict(X))
mse_flexivel = mean_squared_error(y, flexivel.predict(padronizado))

erro_treino = pd.Series(
    {"plano": mse_plano, "k-NN flexível (k=3)": mse_flexivel}
).sort_values()
erro_treino.round(2)

Sobre os mesmos trinta pontos que o ajustaram, o k-NN flexível erra menos que o plano: 36,69 de erro quadrático médio contra 46,48. A superfície que não assume forma nenhuma encosta mais perto do dado observado do que o plano, rígido, consegue chegar — é exatamente a vantagem que a flexibilidade promete.

Só que a mesma flexibilidade que aproxima o ajuste do dado observado pode ir longe demais. Levando o k-NN ao extremo — prever com um único vizinho, em vez da média de três:

In [ ]:
memoriza = KNeighborsRegressor(n_neighbors=1).fit(padronizado, y)
round(mean_squared_error(y, memoriza.predict(padronizado)), 2)

o erro sobre o próprio dado de treino cai a exatamente zero. Faz sentido: com um vizinho só, a renda prevista para cada pessoa é a renda daquela mesma pessoa — ela é o seu próprio vizinho mais próximo, a distância zero. O modelo não aprendeu nada sobre como escolaridade e senioridade se relacionam com renda; decorou as trinta respostas que já tinha. Essa armadilha tem nome — *overfitting* —, e é o preço que a flexibilidade cobra quando nada a segura: ajustar tão perto do dado observado que o modelo passa a repetir o ruído daquela amostra específica, em vez da relação que a gerou. A seção 7.6 mede isso a sério, contra dado que o modelo não viu durante o ajuste; aqui fica só o nome e o sintoma.

> **🔷 Conceito**
>
> | | Paramétrico | Não paramétrico |
> |---|---|---|
> | Forma de `f` | assumida antes de ver o dado (reta, plano, ...) | não assumida — o dado decide |
> | O que se estima | um número fixo de parâmetros | a superfície inteira |
> | Quanto dado exige | pouco | muito mais |
> | Risco principal | a forma errada, que nenhum dado corrige | decorar o ruído da amostra (*overfitting*) |

### A rigidez de um contra a flexibilidade do outro

A diferença fica mais clara em figura do que em número: as mesmas trinta pessoas, com a superfície que cada ajuste prevê para qualquer combinação de escolaridade e senioridade, não só as observadas. Os pontos da malha abaixo também precisam de padronização antes de passar pelo k-NN — e usam a média e o desvio padrão do próprio treino (`X.mean()`, `X.std()`), nunca recalculados sobre a malha, porque é essa régua, e só essa, que o ajuste aprendeu.

In [ ]:
# Figura: Renda prevista para toda combinação de escolaridade e senioridade em `Income2`, pelo plano paramétrico (esquerda) e pelo k-NN flexível com k=3 (direita). Os pontos são as trinta pessoas observadas. As faixas retas e paralelas do plano não têm como acompanhar cada aglomerado de pontos; a superfície flexível dobra ao redor deles.
grade_escolaridade = np.linspace(X["escolaridade"].min(), X["escolaridade"].max(), 60)
grade_senioridade = np.linspace(X["senioridade"].min(), X["senioridade"].max(), 60)
malha_e, malha_s = np.meshgrid(grade_escolaridade, grade_senioridade)
grade = pd.DataFrame({"escolaridade": malha_e.ravel(), "senioridade": malha_s.ravel()})
grade_padronizada = (grade - X.mean()) / X.std()

superficie_plano = plano.predict(grade).reshape(malha_e.shape)
superficie_flexivel = flexivel.predict(grade_padronizada).reshape(malha_e.shape)

niveis = np.linspace(
    min(superficie_plano.min(), superficie_flexivel.min()),
    max(superficie_plano.max(), superficie_flexivel.max()),
    13,
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))

ax1.contourf(malha_e, malha_s, superficie_plano, levels=niveis, cmap="Blues")
ax1.scatter(X["escolaridade"], X["senioridade"], color="C1", s=18, edgecolor="white", linewidth=0.6)
ax1.set_xlabel("escolaridade")
ax1.set_ylabel("senioridade")
ax1.set_title("paramétrico: o plano")

mapa = ax2.contourf(malha_e, malha_s, superficie_flexivel, levels=niveis, cmap="Blues")
ax2.scatter(X["escolaridade"], X["senioridade"], color="C1", s=18, edgecolor="white", linewidth=0.6)
ax2.set_xlabel("escolaridade")
ax2.set_ylabel("senioridade")
ax2.set_title("não paramétrico: k-NN (k=3)")

barra = fig.colorbar(mapa, ax=[ax1, ax2], shrink=0.85, pad=0.02)
barra.set_label("renda prevista")

plt.show()

O plano nunca se curva, e é exatamente por isso que ele erra pouco: três números não têm como acompanhar cada solavanco do dado, só a tendência geral. O k-NN se curva ao redor de cada aglomerado de pontos, e é por isso que ele decora — sem outra coisa dizendo até onde ir, nada o impede de seguir cada ponto isoladamente até o extremo que os números acima já mostraram. Essa flexibilidade tem outro preço, além do dado que ela exige: um modelo que se adapta tão de perto ao que observou fica mais difícil de ler — não sobra um coeficiente único para dizer "mais um ano de escolaridade vale tanto de renda". É esse compromisso, entre acertar mais e entender menos, que a próxima seção discute.

## Precisão contra Interpretabilidade

> **📌 Nota**
>
> Esta seção corresponde à seção 2.1.3 de James et al. (2023).

A seção anterior fechou apontando um preço que a flexibilidade cobra além do dado que ela exige: um modelo que se adapta tão de perto ao que observou fica mais difícil de ler — não sobra um coeficiente único para dizer quanto vale cada preditor. Esse preço não é acidente de um exemplo: é um padrão que atravessa o aprendizado estatístico inteiro, e vale a pena nomear com precisão antes de seguir adiante.

In [ ]:
import matplotlib.pyplot as plt

plt.style.use("estilo-figuras.mplstyle")

### O compromisso: acertar mais, entender menos

De um lado da escolha ficam os métodos rígidos: a regressão linear que a seção anterior ajustou a `Income2` reduz a superfície inteira a um punhado de coeficientes, e é justamente por ter tão pouco a ajustar que cada um deles se lê de cara — "mais um ano de escolaridade vale tanto de renda". Do outro lado ficam os métodos flexíveis: o k-NN não guarda coeficiente nenhum para apontar, só uma vizinhança que muda de ponto a ponto; levado ao extremo — um único vizinho, em vez de vários —, ele nem chega a aprender relação nenhuma, e decora as respostas que já tinha, como a seção anterior mostrou. Descrever o que um k-NN "aprendeu" sobre a relação entre escolaridade, senioridade e renda exige desenhar a superfície inteira, não escrever uma frase.

O padrão geral é esse: quanto mais flexível o método, mais formas de `f` ele consegue acompanhar — e por isso costuma prever melhor —, e menos ele se deixa resumir em algo que uma pessoa lê e repete. Não é uma lei sem exceção, mas é forte o bastante para orientar a escolha de método antes mesmo de olhar o dado: acertar mais e entender menos costumam vir no mesmo pacote.

### Por que escolher o menos flexível de propósito

Se a flexibilidade costuma vir com mais precisão, por que alguém abriria mão dela? Duas razões, e nenhuma das duas é conservadorismo.

A primeira é a pergunta que se está fazendo. A seção 7.2 distinguiu predição de inferência: prever só pede que `Ŷ` saia perto de `Y`; inferir pede entender a forma de `f` — que preditor pesa mais, em que direção, com que força. Um método flexível pode prever tão bem quanto promete e ainda assim não servir para essa segunda pergunta: se a única forma de descrever o que ele fez é reproduzir o código inteiro, não há como apontar nele "isto é o que empurra a resposta para cima". Quando a pergunta é de inferência, um modelo que acerta e não se deixa explicar simplesmente não responde ao que foi perguntado — por melhor que seja a precisão dele.

A segunda é a quantidade de dado disponível. A seção anterior também mostrou o preço que a flexibilidade cobra em dado: pouco dado não basta para uma superfície livre aprender a relação verdadeira, e o que ela aprende no lugar é o ruído específico daquela amostra pequena — a armadilha que ali ficou batizada de *overfitting*. Com pouco dado, o método flexível não chega nem a entregar a precisão que prometia: erra menos no que já viu e mais no que ainda vai ver. Um método rígido, nesse caso, não é a segunda opção por cautela — pode ser a única que entrega alguma precisão de verdade.

> **🔷 Conceito**
>
> | Escolher o **menos** flexível de propósito | Por quê |
> |---|---|
> | A pergunta é de inferência | um ajuste que não se lê não responde "o que explica a resposta", mesmo acertando |
> | O dado é pouco | o método flexível não tem com que aprender a relação — decora a amostra em vez disso |

### O mapa: flexibilidade contra interpretabilidade

Colocando lado a lado alguns dos métodos que este material ainda vai ver — regressão linear, lasso, árvores de decisão, ensembles como bagging e boosting, k-vizinhos mais próximos e redes neurais —, o padrão desta seção aparece como duas pontas de um mesmo eixo: de um lado, métodos rígidos o bastante para se ler o efeito de cada preditor; do outro, métodos livres o bastante para acompanhar qualquer forma que o dado tenha.

Não há dado nenhum por trás do mapa a seguir — nenhum experimento mediu "flexibilidade" ou "interpretabilidade" em unidade nenhuma. É uma comparação relativa entre famílias de método, do jeito que a literatura de aprendizado estatístico costuma desenhar: útil para orientar uma escolha antes de ajustar qualquer coisa, não um número a se citar depois.

In [ ]:
# Figura: Flexibilidade contra interpretabilidade — um mapa qualitativo, sem unidade em nenhum dos dois eixos: a posição de cada método é uma comparação relativa entre famílias, não uma coordenada medida sobre dado nenhum.
metodos = {
    "lasso": (0.08, 0.92),
    "regressão linear": (0.22, 0.80),
    "árvore de decisão": (0.45, 0.55),
    "k-vizinhos\nmais próximos": (0.58, 0.38),
    "bagging e boosting": (0.72, 0.28),
    "redes neurais": (0.92, 0.10),
}

fig, ax = plt.subplots(figsize=(7, 5.2))
for nome, (flexibilidade, interpretabilidade) in metodos.items():
    ax.scatter(flexibilidade, interpretabilidade, color="C0", s=70, zorder=3)
    ax.annotate(
        nome,
        (flexibilidade, interpretabilidade),
        textcoords="offset points",
        xytext=(8, 6),
        fontsize=9,
    )

ax.set_xlim(0, 1.05)
ax.set_ylim(0, 1.05)
ax.set_xticks([0.05, 1.0])
ax.set_xticklabels(["baixa", "alta"])
ax.set_yticks([0.05, 1.0])
ax.set_yticklabels(["baixa", "alta"])
ax.set_xlabel("flexibilidade (ordem qualitativa, sem escala)")
ax.set_ylabel("interpretabilidade (ordem qualitativa, sem escala)")
plt.tight_layout()
plt.show()

Lasso e regressão linear ficam no canto rígido e legível. Redes neurais e ensembles como bagging e boosting ficam no canto oposto, onde mora a precisão que costumam entregar. Árvore de decisão e k-vizinhos mais próximos ficam no meio — nem tão fechados quanto uma reta, nem tão opacos quanto uma rede ou um ensemble —, e é essa posição relativa, não um valor fixo, que este material vai reencontrar quando cada método ganhar o espaço que merece.

Nenhum desses métodos é explicado aqui — cada um tem, à frente, o capítulo que lhe cabe. O que fica desta seção é a posição relativa e o motivo para levá-la a sério: antes de escolher qual construir, vale perguntar que pergunta se está fazendo e quanto dado se tem, porque as duas respostas empurram a escolha para lados opostos do mesmo mapa.

Flexibilidade e interpretabilidade não são o único par de eixos que decide qual método usar. Há uma pergunta anterior a essa: existe uma resposta para aprender contra, ou só um conjunto de preditores sem rótulo nenhum? É essa distinção — entre aprendizado supervisionado e não supervisionado — que a próxima seção faz.

## Supervisionado e Não Supervisionado

> **📌 Nota**
>
> Esta seção corresponde às seções 2.1.4 e 2.1.5 de James et al. (2023).

Toda seção deste capítulo até aqui partiu do mesmo formato de dado, sem nomeá-lo: um `X` que se mede e um `Y` que se quer prever ou explicar — `vendas` em `Advertising`, `renda` em `Income1` e em `Income2`. Essa suposição tem nome, e é ela que separa o aprendizado estatístico em duas famílias.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.style.use("estilo-figuras.mplstyle")

### Uma resposta para cada observação, ou nenhuma

No aprendizado **supervisionado**, cada observação chega como um par (`X_i`, `Y_i`): um vetor de preditores e uma resposta associada a ele, que torna possível checar o quanto uma estimativa `f̂` acerta. É esse par que sustentou a seção 7.2 inteira — estimar `f`, decompor o erro em redutível e irredutível, perguntar se o que interessa é prever `Y` ou entender como `X` o explica. Sem `Y`, nenhuma dessas perguntas tem como ser respondida: não sobra nada para comparar contra a previsão.

No aprendizado **não supervisionado**, cada observação chega só como `X_i` — os preditores, sem resposta nenhuma emparelhada. Não é descuido de quem coletou o dado: para muitos problemas reais simplesmente não existe um `Y` a registrar, nenhum rótulo verdadeiro contra o qual comparar. E a ausência de `Y` muda a natureza da pergunta, não só a resposta disponível: deixa de ser "o que prevê `Y`" e passa a ser "que estrutura existe neste `X`" — quantos grupos naturais o dado tem, quais variáveis se movem juntas, qual observação foge do padrão das demais.

### A mesma nuvem, duas perguntas

A diferença fica mais clara olhando o mesmo dado das duas formas. A nuvem a seguir é simulada — três grupos de pontos em duas dimensões, cada um sorteado ao redor de um centro diferente:

In [ ]:
rng = np.random.default_rng(7)
n_por_grupo = 40
centros = np.array([[0.0, 0.0], [4.5, 4.0], [-1.0, 5.5]])
X = np.concatenate(
    [rng.normal(loc=centro, scale=1.0, size=(n_por_grupo, 2)) for centro in centros]
)
rotulo = np.repeat(np.arange(len(centros)), n_por_grupo)
X.shape, rotulo.shape

In [ ]:
np.unique(rotulo, return_counts=True)

Cento e vinte pontos ao todo, quarenta de cada um dos três grupos que geraram a nuvem. É a mesma nuvem que aparece nos dois painéis a seguir — só muda o que se sabe sobre ela:

In [ ]:
# Figura: A mesma nuvem simulada, olhada de duas formas. Esquerda: cada ponto colorido pelo grupo que o gerou — o problema é supervisionado, porque existe um rótulo contra o qual checar qualquer resposta. Direita: os mesmos 120 pontos, sem cor nenhuma — o problema é não supervisionado, e a pergunta passa a ser quantos grupos existem aqui.
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))

for grupo in range(len(centros)):
    pontos_do_grupo = X[rotulo == grupo]
    ax1.scatter(pontos_do_grupo[:, 0], pontos_do_grupo[:, 1], label=f"grupo {grupo}")
ax1.set_title("supervisionado: o rótulo é conhecido")
ax1.set_xlabel("x1")
ax1.set_ylabel("x2")
ax1.legend()

ax2.scatter(X[:, 0], X[:, 1], color="0.5", label="quantos grupos existem aqui?")
ax2.set_title("não supervisionado: sem rótulo")
ax2.set_xlabel("x1")
ax2.set_ylabel("x2")
ax2.legend()

plt.tight_layout()
plt.show()

À esquerda, a cor de cada ponto vem do grupo que o sorteou — é isso que torna o problema supervisionado: existe uma resposta contra a qual qualquer partição proposta pode ser conferida, certa ou errada. À direita, a mesma nuvem, sem nenhuma cor: o que resta é a pergunta que dá título ao painel. Um método de agrupamento poderia propor uma partição parecida com a da esquerda — e, olhando a nuvem, três grupos parecem mesmo a resposta razoável —, mas não há como confirmar isso contra um rótulo verdadeiro, porque nenhum rótulo verdadeiro existe do lado direito. O que existe é a estrutura que os pontos sugerem por si.

### Regressão contra classificação: a natureza de Y

Dentro do lado supervisionado ainda existe uma segunda distinção — não sobre ter ou não ter `Y`, mas sobre que tipo de valor `Y` assume. Quando `Y` é **quantitativo** — um número que mede algo, como `vendas` em `Advertising` ou `renda` em `Income1` e `Income2` —, o problema é de **regressão**: sem nomear, cada seção deste capítulo até aqui foi um problema de regressão. Quando `Y` é **qualitativo** — uma categoria, como "inadimplente" ou "não inadimplente", "spam" ou "não spam" —, o problema é de **classificação**: não existe meio caminho entre duas categorias, e o erro passa a se medir contando acerto e erro, não medindo distância.

Essa segunda distinção, porém, é menos nítida do que o parágrafo anterior sugere: alguns métodos não escolhem um lado. O k-NN que a seção 7.3 ajustou a `Income2` mediu a distância até os vizinhos mais próximos de cada ponto e devolveu a média das respostas deles — um número, porque ali `Y` era quantitativo. O mesmo mecanismo, aplicado a um `Y` qualitativo, devolve a categoria mais comum entre os vizinhos, em vez da média: nada no método muda, só o que se faz com a vizinhança encontrada. k-NN serve aos dois lados dessa distinção, e não é o único método capaz disso — a fronteira entre regressão e classificação separa problemas, não separa métodos.

As duas próximas seções seguem essa costura: a seção 7.6 mede a qualidade de um ajuste de regressão, com o erro quadrático médio; a seção 7.7 faz o mesmo do lado da classificação, com a taxa de erro e um piso que nenhum classificador consegue furar. Mais adiante neste material, a costura muda de novo, para o aprendizado não supervisionado — onde a nuvem cinzenta desta seção ganha um bloco inteiro dedicado a responder, com método, à pergunta que aqui ficou em aberto.

## Qualidade do Ajuste e o Compromisso Viés-Variância

> **📌 Nota**
>
> Esta seção corresponde às seções 2.2.1 e 2.2.2 de James et al. (2023).

A seção anterior prometeu medir a qualidade de um ajuste de regressão a sério, com o erro quadrático médio. Falta dizer uma coisa antes de medir qualquer coisa: **onde** esse erro é medido decide se o número significa alguma coisa.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.interpolate import make_smoothing_spline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

plt.style.use("estilo-figuras.mplstyle")

### O erro quadrático médio, e a pergunta que interessa

Dado um ajuste $\hat f$ sobre $n$ pares observados, o erro quadrático médio resume o quanto ele erra:

$$
\text{MSE} = \frac{1}{n}\sum_{i=1}^{n} \left(y_i - \hat f(x_i)\right)^2
$$

A seção 7.2 decompôs o erro esperado de qualquer previsão em uma parte redutível — a distância entre $\hat f$ e a $f$ verdadeira, que um ajuste melhor encolhe — e uma parte irredutível, $\mathrm{Var}(\epsilon)$, que nenhum ajuste toca. O MSE tenta medir essa primeira parte, só que de um jeito que engana se medido no lugar errado: se $x_i$ e $y_i$ são os mesmos pontos que ajustaram $\hat f$, nada impede que $\hat f$ passe exatamente por cada um deles, sem ter aprendido nada sobre a relação entre $X$ e $Y$ — só decorado as respostas que já tinha. A seção 7.3 já deu nome a esse sintoma: *overfitting*, o ajuste que persegue o ruído de uma amostra específica em vez da relação que a gerou. Um MSE de treino baixo não distingue essas duas coisas: um ajuste que aprendeu a relação e um ajuste que decorou a amostra podem chegar ao mesmo número ali.

A pergunta que interessa nunca é "quão perto $\hat f$ chega dos pontos que o ajustaram", e sim "quão perto $\hat f$ chega de um ponto que ele nunca viu". Responder essa segunda pergunta exige guardar parte do dado de fora do ajuste, só para medir contra ela depois.

### Separando treino e teste: `train_test_split`

`train_test_split`, do `scikit-learn`, faz exatamente essa separação: reserva uma fração do dado como **teste** — nunca usada para ajustar nada, só para medir depois — e deixa o resto como **treino**. A partir daqui, todo ajuste deste material passa por essa divisão antes de qualquer MSE ser calculado.

### Uma f exata, porque desta vez o dado é simulado

Ainda falta uma peça para tornar a comparação honesta: mesmo com treino e teste separados, o MSE de teste mede a distância até $Y$, não até $f$ — e $Y$ carrega o ruído $\epsilon$ que nenhum ajuste alcança. Para ver o quanto da flexibilidade de um ajuste é real, e não só o ajuste perseguindo esse ruído, seria preciso conhecer a própria $f$ que gerou o dado — o que a seção 7.2 já observou ser impossível com `Advertising` ou mesmo com `Income1`, onde só uma estimativa suave de $f$ estava disponível. Um dado **simulado**, gerado por uma função escolhida por quem escreve, resolve isso: a $f$ verdadeira deixa de ser estimada e passa a ser conhecida, número por número.

In [ ]:
def f_verdadeiro(x):
    return 4.0 + 0.3 * x + 3.0 * np.sin(x)

rng = np.random.default_rng(7)
n = 300
ruido_padrao = 1.5

x = np.sort(rng.uniform(0, 10, size=n))
ruido = rng.normal(0, ruido_padrao, size=n)
y = f_verdadeiro(x) + ruido

x_treino, x_teste, y_treino, y_teste = train_test_split(
    x, y, test_size=0.3, random_state=7
)
x_treino.shape, x_teste.shape

Trezentos pontos, gerados por uma função conhecida — soma de uma tendência linear com uma oscilação de seno — mais um ruído normal com desvio padrão `ruido_padrao = 1.5`, também escolhido, portanto também conhecido. `train_test_split` divide os trezentos em 210 de treino e 90 de teste, na proporção `test_size=0.3` pedida. `random_state=7` fixa qual ponto cai em cada lado — é a mesma semente `7` do capítulo inteiro, só que passada do jeito que esta função em particular pede: um inteiro, não o `rng = np.random.default_rng(7)` usado para gerar `x` e `ruido` duas linhas acima. As duas sementes fixam sorteios diferentes, e as duas precisam estar fixas para o capítulo reproduzir sempre a mesma figura.

`ruido_padrao` não é um detalhe descartável: é o desvio padrão de $\epsilon$, e $\text{Var}(\epsilon) = \text{ruido\_padrao}^2$ é exatamente o piso irredutível que a seção 7.2 descreveu sem poder desenhar. Aqui dá para desenhar, porque foi quem escreveu este material que escolheu o número.

Os 90 pontos de teste bastam para o que vem a seguir: comparar três ajustes específicos entre si. Não bastam para a varredura da próxima seção, que precisa medir com precisão se uma curva cruza ou não o piso — e volta a esse ponto quando chegar lá.

### Três ajustes, três flexibilidades

Com $f$ verdadeira conhecida, dá para comparar ajustes de rigidez bem diferente contra ela, não só contra o dado. O mais rígido dos três é uma reta — `LinearRegression`, a mesma ferramenta que a seção 7.3 usou em `Income2`. Os outros dois são *smoothing splines*: uma curva suave, ajustada minimizando a soma dos resíduos ao quadrado mais uma penalidade sobre a curvatura da própria curva, controlada por um parâmetro $\lambda$. Quanto maior $\lambda$, mais a curvatura pesa contra o ajuste e mais a curva se aproxima de uma reta; quanto menor $\lambda$, menos a curvatura pesa, e mais livre a curva fica para dobrar atrás de cada ponto.

In [ ]:
ordem = np.argsort(x_treino)
x_treino_ordenado = x_treino[ordem]
y_treino_ordenado = y_treino[ordem]

reta = LinearRegression().fit(x_treino.reshape(-1, 1), y_treino)
spline_moderado = make_smoothing_spline(x_treino_ordenado, y_treino_ordenado, lam=0.5)
spline_flexivel = make_smoothing_spline(
    x_treino_ordenado, y_treino_ordenado, lam=0.0001
)

resultado = pd.DataFrame(
    {
        "treino": [
            mean_squared_error(y_treino, reta.predict(x_treino.reshape(-1, 1))),
            mean_squared_error(y_treino, spline_moderado(x_treino)),
            mean_squared_error(y_treino, spline_flexivel(x_treino)),
        ],
        "teste": [
            mean_squared_error(y_teste, reta.predict(x_teste.reshape(-1, 1))),
            mean_squared_error(y_teste, spline_moderado(x_teste)),
            mean_squared_error(y_teste, spline_flexivel(x_teste)),
        ],
    },
    index=["reta", "spline moderado (λ=0,5)", "spline muito flexível (λ=0,0001)"],
)
resultado.round(2)

In [ ]:
# Figura: Trezentos pontos simulados a partir de uma f conhecida (linha azul), divididos em treino (círculos) e teste (triângulos), com três ajustes de flexibilidade crescente: a reta mal acompanha a curva; o spline muito flexível dobra atrás de cada ponto de treino, inclusive os que são só ruído.
grade = np.linspace(x.min(), x.max(), 400)

fig, ax = plt.subplots()
ax.scatter(x_treino, y_treino, s=14, alpha=0.5, label="treino")
ax.scatter(x_teste, y_teste, s=20, alpha=0.6, marker="^", label="teste")
ax.plot(grade, f_verdadeiro(grade), linewidth=2.5, label="f verdadeiro")
ax.plot(grade, reta.predict(grade.reshape(-1, 1)), linewidth=2, label="reta")
ax.plot(grade, spline_moderado(grade), linewidth=2, label="spline moderado")
ax.plot(grade, spline_flexivel(grade), linewidth=2, label="spline muito flexível")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.legend(fontsize=8, ncols=2)
plt.tight_layout()
plt.show()

A reta erra 5,83 de MSE no treino e 5,28 no teste: rígida demais para acompanhar a oscilação do seno, ela erra parecido nos dois lados, porque a forma que assumiu — reta — está errada em qualquer amostra, treino ou teste. O spline muito flexível é o oposto: 1,05 no treino, quase decorando cada ponto, contra 2,77 no teste — o ajuste que mais se aproxima do dado observado é o que mais piora quando confrontado com dado novo. O spline moderado fica entre os dois e vence os dois no teste: 1,66 no treino, 2,31 no teste — nem tão rígido quanto a reta, nem tão perseguidor do ruído quanto o spline muito flexível.

Repare que o MSE de **treino** só cai conforme a flexibilidade cresce — 5,83, depois 1,66, depois 1,05, estritamente decrescente —, enquanto o de **teste** cai e depois sobe: 5,28, depois 2,31, depois 2,77. É exatamente esse formato, treino sempre caindo e teste em U, que a próxima figura mede com uma varredura bem mais fina do que só três pontos.

### A curva em U, e o piso que ela não fura

Três ajustes já bastam para sugerir o formato em U do MSE de teste; uma varredura em $\lambda$, do mais rígido ao mais flexível, mostra a curva inteira. Só que o MSE de teste, medido sobre poucos pontos, é ele mesmo uma variável aleatória — com os 90 pontos já separados, ele pode cair abaixo do piso irredutível por sorte da amostra, sem que o piso tenha sido furado de verdade. O próprio James et al. (2023) evita esse problema medindo a curva de teste sobre um conjunto de teste bem maior; a mesma simulação que dá acesso a $f$ permite gerar um.

In [ ]:
n_teste_grande = 5000
x_teste_grande = rng.uniform(0, 10, size=n_teste_grande)
ruido_teste_grande = rng.normal(0, ruido_padrao, size=n_teste_grande)
y_teste_grande = f_verdadeiro(x_teste_grande) + ruido_teste_grande
x_teste_grande.shape

Cinco mil pontos novos, da mesma $f$ e do mesmo `ruido_padrao` — nunca usados para ajustar nada, só para medir a varredura a seguir com a precisão que 90 pontos não entregam.

In [ ]:
lambdas = np.logspace(3, -5, 60)
mse_treino = np.array(
    [
        mean_squared_error(
            y_treino, make_smoothing_spline(x_treino_ordenado, y_treino_ordenado, lam=l)(x_treino)
        )
        for l in lambdas
    ]
)
mse_teste = np.array(
    [
        mean_squared_error(
            y_teste_grande,
            make_smoothing_spline(x_treino_ordenado, y_treino_ordenado, lam=l)(x_teste_grande),
        )
        for l in lambdas
    ]
)

piso = ruido_padrao**2
indice_minimo = int(np.argmin(mse_teste))
lambda_minimo = lambdas[indice_minimo]
mse_teste_minimo = mse_teste[indice_minimo]

treino_sempre_cai = bool(np.all(np.diff(mse_treino) <= 0))
teste_nunca_fura_piso = bool(np.all(mse_teste >= piso))

piso, round(float(lambda_minimo), 2), round(float(mse_teste_minimo), 2), treino_sempre_cai, teste_nunca_fura_piso

In [ ]:
# Figura: MSE de treino e de teste (este sobre os 5.000 pontos do conjunto de teste grande) contra a flexibilidade do spline (λ decrescente, escala log). O treino cai sem parar; o teste cai, toca um mínimo perto de λ=0,30 e volta a subir. A linha tracejada é Var(ε) — o piso irredutível que este material conhece porque gerou o próprio ruído — e a curva de teste nunca desce abaixo dela.
fig, ax = plt.subplots()
ax.plot(lambdas, mse_treino, linewidth=2, label="MSE treino")
ax.plot(lambdas, mse_teste, linewidth=2, label="MSE teste")
ax.axhline(piso, linestyle="--", linewidth=1.5, label="piso irredutível (Var(ε))")
ax.set_xscale("log")
ax.invert_xaxis()
ax.set_xlabel("λ do spline (escala log; menor λ = mais flexível →)")
ax.set_ylabel("MSE")
ax.legend()
plt.tight_layout()
plt.show()

Ao longo dos 60 valores de $\lambda$ varridos, o MSE de treino cai sem interrupção — confirmado pelo chunk acima, `treino_sempre_cai` é `True`. O de teste não: desce de um extremo rígido, toca o mínimo, 2,41, perto de $\lambda=0,30$ — na mesma vizinhança do spline moderado ($\lambda=0,5$), escolhido às cegas na seção anterior, sem estar exatamente no mesmo ponto — e volta a subir do outro lado, onde a flexibilidade passa a perseguir ruído. Em nenhum dos 60 pontos, medidos contra os cinco mil pontos do conjunto de teste grande, o MSE de teste desce abaixo de `piso = 2.25`, a variância do ruído que este material mesmo gerou — `teste_nunca_fura_piso` é `True`. É o piso que a seção 7.2 descreveu sem poder mostrar: nenhuma flexibilidade, por maior que seja, empurra o erro esperado abaixo da parte do problema que o próprio ajuste não controla.

### A decomposição viés-variância

A curva em U tem uma explicação, e ela vem de decompor o MSE esperado de teste, num ponto novo $x_0$, em três parcelas:

$$
E\left[\left(y_0 - \hat f(x_0)\right)^2\right] = \mathrm{Var}\left(\hat f(x_0)\right) + \left[\mathrm{Bias}\left(\hat f(x_0)\right)\right]^2 + \mathrm{Var}(\epsilon)
$$

**Viés** é o erro que vem da forma escolhida ser rígida demais para a relação verdadeira — o preço que a reta paga por ser reta, mesmo com todo o dado de treino do mundo à disposição. **Variância** é o quanto $\hat f$ mudaria se reajustada sobre outra amostra de treino, sorteada da mesma população — o preço que o spline muito flexível paga por seguir de perto os pontos específicos que calhou de ver: um pouco de ruído a mais ou a menos nesses pontos, e a curva inteira se reacomoda atrás deles. **Var(ε)** é o piso que a seção anterior desenhou, e nenhuma das duas parcelas anteriores toca.

Os três ajustes desta seção ocupam três lugares nesse espaço. A reta tem viés alto — nenhuma reta acompanha a curvatura do seno — e variância baixa: trocar a amostra de treino move pouco o coeficiente de uma reta. O spline muito flexível inverte os dois: viés baixo, porque ele consegue seguir qualquer curvatura, e variância alta, porque essa mesma liberdade o deixa refém do ruído específico da amostra que recebeu. O spline moderado não zera nenhuma das duas parcelas — nenhum ajuste zera —, mas encontra a combinação das duas que soma menos, e é essa soma, viés ao quadrado mais variância, mais o piso que nenhum dos dois toca, que a curva em U traça ao percorrer a flexibilidade.

A mesma pergunta, do lado da classificação, ainda precisa de resposta: lá o erro não se mede em distância, e sim contando acerto e erro, e existe um piso análogo a este — o classificador de Bayes. É para lá que a próxima seção vai.

## Classificação e o Classificador de Bayes

> **📌 Nota**
>
> Esta seção corresponde à seção 2.2.3 de James et al. (2023).

A seção anterior mediu o ajuste em distância: o quanto $\hat f(x)$ erra de um $y$ numérico. Quando $Y$ é uma classe — doente ou não, spam ou não —, distância deixa de fazer sentido, e a régua muda: o que conta é se a previsão acertou o rótulo ou errou.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.neighbors import KNeighborsClassifier

plt.style.use("estilo-figuras.mplstyle")

### A taxa de erro

O análogo do MSE, para uma resposta qualitativa, é a **taxa de erro**: a fração das previsões que erram o rótulo.

$$
\text{Taxa de erro} = \frac{1}{n}\sum_{i=1}^n \mathbb{1}(y_i \neq \hat y_i)
$$

$\mathbb{1}(y_i \neq \hat y_i)$ vale 1 quando o classificador erra a observação $i$ e 0 quando acerta; a média sobre as $n$ observações é a proporção de erros. A mesma ressalva da seção anterior continua valendo, ponto por ponto: a taxa de erro que interessa é a de **teste**, medida sobre dado que não ajustou o classificador — a de **treino** só diz o quanto o classificador decorou o que já tinha visto, e cai mesmo quando essa memorização não ensinou nada sobre dado novo.

### O classificador de Bayes

Existe uma regra que minimiza a taxa de erro esperada, e ela é simples de enunciar: para cada ponto $x_0$, atribuir a classe mais provável dado $X = x_0$.

$$
\hat C(x_0) = \operatorname*{argmax}_{j} \; P(Y = j \mid X = x_0)
$$

Nenhuma outra regra faz melhor — atribuir qualquer classe que não seja a mais provável só pode aumentar a chance de errar aquele ponto. É por isso que essa regra tem nome próprio, **classificador de Bayes**, e sua taxa de erro tem nome próprio também: **taxa de erro de Bayes**, o piso que nenhum classificador fura, o análogo exato do $\mathrm{Var}(\epsilon)$ da seção anterior.

Só que descrever o classificador de Bayes é mais fácil do que construí-lo: ele exige conhecer $P(Y = j \mid X = x_0)$ de verdade, para todo $x_0$ — a distribuição condicional exata de $Y$ dado $X$. Com dado real, ninguém tem essa distribuição; o que existe são $n$ observações, e a distribuição que as gerou permanece desconhecida. É por isso que o classificador de Bayes, fora de um material como este, é inatingível: um padrão contra o qual comparar, nunca um classificador que alguém constrói. Aqui é diferente pelo mesmo motivo da seção anterior — o dado vai ser simulado, e quem simula escolhe $P(Y \mid X)$.

In [ ]:
def p_verdadeiro(x1, x2):
    return 1.0 / (1.0 + np.exp(-(3.0 * np.sin(x1) - x2 + 5.0)))

rng = np.random.default_rng(7)

def gerar(rng, n):
    x1 = rng.uniform(0, 10, size=n)
    x2 = rng.uniform(0, 10, size=n)
    p = p_verdadeiro(x1, x2)
    y = (rng.uniform(size=n) < p).astype(int)
    return np.column_stack([x1, x2]), y

X_treino, y_treino = gerar(rng, 300)
X_teste, y_teste = gerar(rng, 20_000)
X_treino.shape, X_teste.shape

Dois preditores, $X_1$ e $X_2$, uniformes em $[0, 10]$; a probabilidade condicional de $Y=1$ é uma logística sobre uma combinação deles — tendência linear em $x_2$ mais uma oscilação de seno em $x_1$, a mesma forma de combinar reta e seno que gerou $f$ na seção anterior:

$$
P(Y = 1 \mid X = x) = \frac{1}{1 + e^{-(3\sin(x_1) - x_2 + 5)}}
$$

O rótulo $Y$ sai de um sorteio com essa probabilidade — `rng.uniform(size=n) < p` é exatamente um Bernoulli$(p)$. Trezentos pontos para treino, na mesma ordem de grandeza da seção anterior; vinte mil para teste, maior do que os cinco mil de lá, e a razão para o tamanho vem mais adiante, quando o teto de Bayes precisar ser medido com precisão contra o k-NN.

A fronteira de Bayes — onde $P(Y=1\mid X=x)=1/2$, e as duas classes empatam — fica onde o argumento da logística zera: $x_2 = 3\sin(x_1) + 5$, uma curva, não uma reta. Fora dessa curva, uma classe é sempre mais provável que a outra, mas nunca com certeza: em quase todo ponto do domínio, $P(Y=1\mid X=x)$ fica entre 0 e 1, nunca exatamente em 0 ou 1 — há sempre uma chance da classe menos provável aparecer, e é essa sobreposição que faz a taxa de erro de Bayes ser maior que zero.

In [ ]:
grade_integral = np.linspace(0, 10, 4001)
G1, G2 = np.meshgrid(grade_integral, grade_integral)
Pg = p_verdadeiro(G1, G2)
piso_bayes = float(np.mean(np.minimum(Pg, 1 - Pg)))

pred_bayes_teste = (p_verdadeiro(X_teste[:, 0], X_teste[:, 1]) > 0.5).astype(int)
erro_bayes_teste = float(np.mean(pred_bayes_teste != y_teste))

round(piso_bayes, 4), round(erro_bayes_teste, 4)

A taxa de erro de Bayes não precisa de nenhum sorteio para ser calculada: em cada ponto $x$, o classificador de Bayes ainda erra com probabilidade $\min\big(P(Y{=}1\mid X{=}x),\, 1 - P(Y{=}1\mid X{=}x)\big)$ — a chance da classe que ele não escolheu. A média dessa quantidade sobre o domínio inteiro, calculada numa grade fina, dá **13,26%**: o piso exato, porque $P(Y\mid X)$ é conhecida ponto a ponto. Aplicar a mesma regra de Bayes aos 20.000 pontos de teste — prever a classe mais provável em cada um, e comparar com o $y$ que de fato saiu do sorteio — dá **13,32%**, quase o mesmo número: a diferença é só o ruído de um sorteio finito em volta do valor exato.

### k-vizinhos mais próximos

O classificador de Bayes conhece $P(Y \mid X)$; o k-NN não conhece nada disso, e estima. Para um ponto $x_0$, ele olha os $k$ pontos de treino mais próximos e vota: a classe que aparece mais entre os vizinhos é a prevista. Essa votação é uma estimativa da própria quantidade que o classificador de Bayes usaria — a fração de vizinhos da classe 1 aproxima $P(Y{=}1\mid X{=}x_0)$ —, só que calculada localmente, com $k$ pontos, em vez de conhecida de antemão.

$k$ decide o quanto essa estimativa é local. Com $k$ pequeno, a vizinhança é minúscula, às vezes um só ponto, e a fronteira persegue cada observação de treino individualmente — variância alta, porque trocar a amostra de treino move a fronteira inteira. Com $k$ grande, a vizinhança cresce até deixar de ser vizinhança: pontos distantes do $x_0$ que está sendo classificado entram na votação, dilutem qualquer estrutura local, e a fronteira endurece até quase virar uma reta — viés alto, o mesmo preço que a reta pagava na seção anterior.

In [ ]:
ks = list(range(1, 300, 4))
erros_treino = []
erros_teste = []
for k in ks:
    modelo = KNeighborsClassifier(n_neighbors=k)
    modelo.fit(X_treino, y_treino)
    erros_treino.append(float(np.mean(modelo.predict(X_treino) != y_treino)))
    erros_teste.append(float(np.mean(modelo.predict(X_teste) != y_teste)))

erros_treino = np.array(erros_treino)
erros_teste = np.array(erros_teste)
indice_minimo = int(np.argmin(erros_teste))
k_minimo = ks[indice_minimo]
teste_minimo = float(erros_teste[indice_minimo])
nunca_abaixo_do_piso = bool(np.all(erros_teste >= piso_bayes))
margem_pp = round((teste_minimo - piso_bayes) * 100, 2)

k_minimo, round(teste_minimo, 4), round(float(erros_treino[0]), 4), nunca_abaixo_do_piso, margem_pp

Setenta e cinco valores de $k$ varridos, de 1 a 297, de quatro em quatro. Em $k=1$, o erro de treino é **0%** — cada ponto de treino é o próprio vizinho mais próximo de si mesmo, então a votação sempre acerta o rótulo que já tinha. O erro de teste em $k=1$ não acompanha: a fronteira que decorou cada ponto de treino erra a vizinhança de um ponto novo com muito mais frequência. O menor erro de teste do varrimento inteiro sai em $k=9$: **15,08%**. Em nenhum dos 75 valores de $k$ a taxa de erro de teste desce abaixo dos 13,26% calculados acima — nem no ponto que mais se aproxima, $k=9$, que fica **1,82** ponto percentual acima (`margem_pp`). Com um conjunto de teste de poucas centenas de pontos, essa distância teria variância grande o bastante para a taxa de teste cruzar o piso por sorte da amostra, como a seção anterior mediu acontecer com o MSE; vinte mil pontos de teste bastam para essa comparação não depender do sorteio.

### A fronteira que cada k desenha

Três valores de $k$, bem separados, mostram o que a tabela de erros já contou em número: $k=1$, o mínimo do varrimento ($k=9$), e $k=199$, perto do outro extremo.

In [ ]:
k_grande = 199
k_tres = [1, k_minimo, k_grande]
modelos_tres = [
    KNeighborsClassifier(n_neighbors=k).fit(X_treino, y_treino) for k in k_tres
]
tabela_k = pd.DataFrame(
    {
        "treino": [float(np.mean(m.predict(X_treino) != y_treino)) for m in modelos_tres],
        "teste": [float(np.mean(m.predict(X_teste) != y_teste)) for m in modelos_tres],
    },
    index=[f"k={k_tres[0]}", f"k={k_tres[1]} (menor erro de teste)", f"k={k_tres[2]}"],
)
tabela_k.round(4)

$k=1$ decora o treino (0% de erro) e erra 20,92% do teste; $k=199$ erra 26,67% do treino e 23,03% do teste, rígido demais para acompanhar a curva de $3\sin(x_1) + 5$; $k=9$ fica entre os dois em treino (14,33%) e vence os dois no teste (15,08%) — nem tão preso ao ruído de cada ponto quanto $k=1$, nem tão achatado quanto $k=199$.

In [ ]:
# Figura: Fronteiras de decisão do k-NN para k=1, k=9 e k=199 (verde), sobre os mesmos 300 pontos de treino coloridos pela classe verdadeira, com a fronteira de Bayes (roxo tracejado) por cima. k=1 dobra atrás de cada ponto; k=199 quase não acompanha a curvatura da fronteira verdadeira; k=9 é o que chega mais perto dela.
grade_fig = np.linspace(0, 10, 200)
Xg, Yg = np.meshgrid(grade_fig, grade_fig)
pontos_grade = np.column_stack([Xg.ravel(), Yg.ravel()])
proba_bayes = p_verdadeiro(Xg, Yg)

fig, eixos = plt.subplots(1, 3, figsize=(12, 4.3), sharex=True, sharey=True)
for ax, k, modelo in zip(eixos, k_tres, modelos_tres):
    proba_knn = modelo.predict_proba(pontos_grade)[:, 1].reshape(Xg.shape)
    ax.scatter(
        X_treino[y_treino == 0, 0], X_treino[y_treino == 0, 1],
        s=12, alpha=0.6, color="#4195D1", label="classe 0",
    )
    ax.scatter(
        X_treino[y_treino == 1, 0], X_treino[y_treino == 1, 1],
        s=12, alpha=0.6, color="#D9480F", label="classe 1",
    )
    ax.contour(Xg, Yg, proba_knn, levels=[0.5], colors="#2F8F46", linewidths=2)
    ax.contour(Xg, Yg, proba_bayes, levels=[0.5], colors="#9775FA", linestyles="--", linewidths=1.5)
    ax.set_title(f"k = {k}")
    ax.set_xlabel("x1")
eixos[0].set_ylabel("x2")
eixos[0].plot([], [], color="#2F8F46", linewidth=2, label="fronteira k-NN")
eixos[0].plot([], [], color="#9775FA", linestyle="--", linewidth=1.5, label="fronteira de Bayes")
eixos[0].legend(fontsize=7, loc="lower left")
plt.tight_layout()
plt.show()

É a mesma curva em U da seção anterior, só que medida em taxa de erro em vez de MSE: $k$ pequeno decora o treino e erra o teste por variância, $k$ grande simplifica demais e erra por viés, e o meio-termo — aqui $k=9$ — não elimina nenhuma das duas parcelas, só encontra a combinação que soma menos, sem nunca descer abaixo do piso que o classificador de Bayes marca. O compromisso é o mesmo; muda só a métrica que o mede.

## Leituras adicionais

*A escrever.*

## Referências

- **James; Witten; Hastie; Tibshirani; Taylor**. *An Introduction to Statistical Learning with Applications in Python*. Springer. 2023.